# FEM loop milestone: change the geometry, the solved values move

Decisions: the milestone is the **iterative loop**, not converged numbers. The chip is one feedline
with two hanging resonators on the Quantcore PDK (sapphire 650 µm, Nb with kinetic inductance, the pad rule,
cheese band). We render it to AEDT 2024 R1, solve it, read the results back and compare them with the LOM. Then we
change the geometry and do it all again.

- **R1**, a λ/4 resonator shorted at its coupler: the usual readout hanger. Its κ is large enough for the driven
  S21 notch to resolve.
- **R2**, a λ/2 resonator open at both ends: a floating conductor, so **Q3D** measures its capacitance to ground
  and its coupling capacitance to the feedline.
- **Rev 1** changes three knobs: R1's coupler goes from 250 to 400 µm (κ_R1 must rise), R2 is 400 µm shorter
  (f_R2 must rise), and R2's coupler goes from 250 to 400 µm (C(R2–FL) must rise).

Everything below is driven by `examples/fem_loop/fem_loop.py`. The command line does the same:
`python examples/fem_loop/fem_loop.py build | solve | report`.

**`AEDT_MODE`** (next to `RUN_FEM`): `"batch"` solves in a new non-graphical AEDT; `"gui"` opens a new AEDT window and leaves it open, so you watch the mesh, the convergence and the fields; `"attach"` runs inside the AEDT window you already have open (or opens one) and leaves the project there. Every mode prints a progress line (step, pass, mesh, ΔF / ΔS / ΔC) every 30 s, so a stuck run is visible; on a timeout only what qcc started is stopped.

In [ ]:
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "examples" / "fem_loop")); sys.path.insert(0, str(ROOT / "examples" / "designs"))
os.environ.setdefault("QCC_LIVE", "0")
from IPython.display import Image, Markdown, display
import fem_loop as L

RUN_FEM = False  # True: render + solve all 6 AEDT jobs again (≈ 1.5 h here)
AEDT_MODE = "batch"  # "batch" (no window) | "gui" (a new AEDT window) | "attach" (yours)

## 1 · The two revisions

Each revision is one `Designer` script (`L.design(rev)`):
- the pads use `edge=` (the PDK pad rule);
- the feedline is a route;
- each resonator is a fixed-length meander from an open free end into a hanger on the feedline.

The hanger is pinned with `near=`. Its length and its type come from the knobs: λ/4 is `end="short"`, λ/2 is
`end="open"`. The resonator lengths come from the PDK CPW, kinetic inductance included
(`analysis.synthesis.quarter_wave_length_um`).

In [ ]:
display(Markdown("| rev | R1 coupler (µm) | R2 coupler (µm) | R2 length change (µm) |\n|---|---|---|---|\n" + "\n".join(
    f"| {rev} | {k['R1']['coupler']:g} | {k['R2']['coupler']:g} | {k['R2']['dl']:+g} |" for rev, k in L.REVS.items())))
from quantcore_circuits.assembly import build
from quantcore_circuits.assembly.drc import check
from quantcore_circuits.live.snapshot import snapshot
for rev in L.REVS:
    d, path = L.design(rev)
    result = build(d.design)
    print(f"rev {rev}: {path.name} —", str(check(result, netlist=False)).splitlines()[0])
    png = snapshot(result, ROOT / "examples" / "fem_loop" / f"fem_loop_rev{rev}.png", size=(1000, 1000))
display(Image(filename=str(png), width=560))

## 2 · LOM prediction

The layout becomes a node-connected circuit (`lom_from_design`), filled with the component estimates and the CPW
formulas (`fill`), then simulated (`simulate`: QuCat normal modes, then QuTiP). For a hanger resonator that gives
f_r and κ_ext.

In [ ]:
lom = {rev: L.lom(L.design(rev)[1]) for rev in L.REVS}
display(Markdown("| quantity | rev 0 | rev 1 |\n|---|---|---|\n" + "\n".join(
    f"| {k} | {lom[0][k]:.4f} | {lom[1][k]:.4f} |" for k in sorted(lom[0]))))

## 3 · Send the jobs to AEDT, bring the results back

`L.solve(rev)` renders and solves three analyses per revision. Each job runs in its own non-graphical batch AEDT,
which never attaches to a window you have open:

| analysis | what it measures | job directory |
|---|---|---|
| `q3d_cmatrix` | Maxwell C-matrix of FL and the floating R2 | `build/jobs/fem_loop_rev<rev>_FL2R_q3d_cmatrix` |
| `eigenmode_epr` | mode frequencies f and Q (the cut feedline is terminated in 50 Ω, so κ = f/Q) | `..._eigenmode_epr` |
| `hfss_driven` | S21 between the two cut ends of the feedline; one notch fit per resonator | `..._hfss_driven` |

The sim group (`FL2R`) selects only the resonators. The box, their bounding box plus 300 µm, cuts the feedline,
and its two ends become the driven ports. The driven run uses the gap mesh seed (`mesh_size={"1t1_gap": 10}`):
without it, the first run's adaptive mesh never refined the resonators and S21 stayed flat. Each job's
`results/results.json` is what `collect_results` read back.

**`AEDT_MODE`** (next to `RUN_FEM`): `"batch"` solves in a new non-graphical AEDT; `"gui"` opens a new AEDT window and leaves it open, so you watch the mesh, the convergence and the fields; `"attach"` runs inside the AEDT window you already have open (or opens one) and leaves the project there. Every mode prints a progress line (step, pass, mesh, ΔF / ΔS / ΔC) every 30 s, so a stuck run is visible; on a timeout only what qcc started is stopped.

In [ ]:
if RUN_FEM:
    for rev in L.REVS:
        L.solve(rev, aedt=AEDT_MODE)
for rev in L.REVS:
    for analysis in L.ANALYSES:
        job = L.JOBS / f"fem_loop_rev{rev}_{L.GROUP}_{analysis}"
        print(f"rev {rev} {analysis:13s}", "results ✓" if (job / "results" / "results.json").exists() else "— not solved", job)

## 4 · LOM vs FEM, rev 0 → rev 1

Every number below comes from a file in the job directories; the table is also written to
`examples/fem_loop/results.md`.

In [ ]:
rows = [L.summary(rev) for rev in L.REVS]
table = L.report(rows)
(ROOT / "examples" / "fem_loop" / "results.md").write_text(table + "\n", encoding="utf-8")
display(Markdown(table))

**Reading it**
- **R2 frequency moves with its length.** All three sources agree on the size of the shift, +0.377 to +0.392 GHz.
  The absolute values agree within 1 %: eigenmode +0.2 %, driven −0.8 % against the LOM.
- **κ_R1 rises with the longer coupler** in all three: ×2.53 in the LOM, ×2.66 in the eigenmode, ×2.77 in the driven
  fit. The LOM's absolute κ is about 35 % below FEM: the hanger k is calibrated on a 300 µm coupler.
- **Q3D sees the longer R2 coupler.** C(R2–FL) goes from −2.7 to −4.5 fF (×1.67, close to the length ratio 1.6), and
  R2's own capacitance drops by 4.7 %, as the 400 µm shorter line should.
- **The LOM underestimates κ for the open-ended λ/2 hanger** (R2) by about 5×. It is a calibration gap: the LOM's
  HANGER k is fitted on shorted λ/4 hangers. It goes on the list for the LOM calibration.
- Convergence: the driven runs converged (ΔS < 0.02). The eigenmode and Q3D runs stopped at their pass limit; the
  non-converged values are accepted for this milestone.

## 5 · Check it yourself in AEDT

Each job directory holds the AEDT project (`*_project.aedt`). Open it in the AEDT 2024 R1 GUI to see the geometry,
mesh, setup and solution. `check_project` reads the saved project back and lists its settings, without AEDT:
stack, metal sheets, ports, nets and every solver setting. `qcc hfss check <job>` prints the same.

In [ ]:
from quantcore_circuits.sim_tools.hfss import check_project
for analysis in L.ANALYSES:
    info = check_project(L.JOBS / f"fem_loop_rev1_{L.GROUP}_{analysis}")
    bad = [c for c in info["checks"] if not c["ok"]]
    print(f"{analysis:13s} project check: {'ok' if info['ok'] else 'ISSUES'} ({len(info['checks'])} checks, "
          f"{len(info['settings'])} settings read back)")
    for c in bad:
        print("   ✗", c["check"], ":", c["detail"])
print("projects:")
for p in sorted(L.JOBS.glob("fem_loop_rev*_FL2R_*/*_project.aedt")):
    print("  ", p)